# Benchmark LifeMate trên Colab
Chọn runtime GPU, chuẩn bị repo và kit TRAIN chính thức trong hai thư mục riêng. Chạy smoke trước khi mở 269 mẫu còn lại.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/content/lifemate_superklang')
KIT = Path('/content/train+public_test')
RUNS = REPO / 'outputs' / 'colab'
MODELS = Path('/content/lifemate_models')
PLANNER = 'planner'  # Use 'fallback_planner' before both arms if the main checkpoint cannot run.

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], cwd=REPO, env=env, check=True)

run('nvidia-smi')
assert (KIT / 'train' / 'train.jsonl').is_file()
RUNS.mkdir(parents=True, exist_ok=True)
run(sys.executable, '-m', 'pip', 'install', '-e', '.[model]')
run(sys.executable, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())')

In [ ]:
from huggingface_hub import snapshot_download

pins = json.loads((REPO / 'configs' / 'colab_models.json').read_text())
for role in (PLANNER, 'asr'):
    snapshot_download(**pins[role], local_dir=MODELS / role,
                      allow_patterns=['*.json', '*.jinja', '*.safetensors', '*.bin', 'merges.txt', 'LICENSE', 'README.md'])

offline = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
catalog = KIT / 'train' / 'train_tools_catalog.json'
smoke_manifest = REPO / 'research' / 'smoke_split.json'

def benchmark(manifest, tag):
    target = RUNS / PLANNER / tag
    gold = target / 'gold.jsonl'
    run(sys.executable, '-B', '-m', 'scripts.replay_train_gold', '--data-root', KIT,
        '--manifest', manifest, '--output', gold, env=offline)
    for mode in ('direct', 'compiled'):
        run(sys.executable, '-B', '-m', 'src.inference', 'official',
            '--split', 'train', '--data-root', KIT, '--train-manifest', manifest,
            '--model-dir', MODELS / PLANNER, '--asr-dir', MODELS / 'asr', '--device', 'cuda',
            '--planner-mode', mode, '--output', target / f'{mode}.jsonl',
            '--train-report', target / f'{mode}.report.json', env=offline)
    run(sys.executable, '-B', '-m', 'scripts.compare_train', 'compare',
        '--manifest', manifest, '--gold', gold, '--catalog', catalog,
        '--direct', target / 'direct.jsonl', '--direct-report', target / 'direct.report.json',
        '--compiled', target / 'compiled.jsonl', '--compiled-report', target / 'compiled.report.json',
        '--output', target / 'comparison.json', env=offline)

In [ ]:
benchmark(smoke_manifest, 'smoke40')

## 269 mẫu ngoài smoke
Chạy sau khi đã chốt checkpoint, prompt và thuật toán từ smoke.

In [ ]:
locked_manifest = RUNS / 'locked269.json'
run(sys.executable, '-B', '-m', 'scripts.compare_train', 'prepare', '--output', locked_manifest)
benchmark(locked_manifest, 'locked269')